In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from PIL import Image
import cv2

In [ ]:
# If running on colab you can download the data using gdown. Uncomment the below code.
import gdown
!gdown 1JTvMkEaR3AxqAagYzfLebnS_RavA7agR #x_train_img.npz
!gdown 1bQxyUXwvUn2cPjb5gIp-jB3CUpqD4Ca1 #x_test_img.npz
!gdown 1tED7-ZgWT0ONtohOD1BCCrlTvXfQM6L0 #x_train.csv
!gdown 1k36wEmeaks1pg2Q5t3aW3e24uxDaUll8 #x_test.csv
!gdown 1J2fEdFSfZnTJ9YwxNnB8qDEOrIOkqbkg #y_train.csv

Downloading...
From (original): https://drive.google.com/uc?id=1JTvMkEaR3AxqAagYzfLebnS_RavA7agR
From (redirected): https://drive.google.com/uc?id=1JTvMkEaR3AxqAagYzfLebnS_RavA7agR&confirm=t&uuid=dbf9ebfe-05d1-41c8-ace2-05f60bdd52f7
To: /content/x_train_img.npz
100% 177M/177M [00:01<00:00, 158MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1bQxyUXwvUn2cPjb5gIp-jB3CUpqD4Ca1
From (redirected): https://drive.google.com/uc?id=1bQxyUXwvUn2cPjb5gIp-jB3CUpqD4Ca1&confirm=t&uuid=4d530b64-b06f-48d2-b129-476208885d37
To: /content/x_test_img.npz
100% 49.0M/49.0M [00:00<00:00, 85.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tED7-ZgWT0ONtohOD1BCCrlTvXfQM6L0
To: /content/x_train.csv
100% 160k/160k [00:00<00:00, 87.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1k36wEmeaks1pg2Q5t3aW3e24uxDaUll8
To: /content/x_test.csv
100% 40.2k/40.2k [00:00<00:00, 102MB/s]
Downloading...
From: https://drive.google.com/uc?id=1J2fEdFSfZnTJ9YwxNnB8qDEOrIOkqbkg
To: /content

In [ ]:
IMG_FILE = "x_train_img.npz"
metadata_file = "x_train.csv"
LABEL_CSV = "y_train.csv"
IMG_TEST_FILE = "x_test_img.npz"

In [ ]:
BATCH_SIZE    = 32
NUM_WORKERS   = 2
PHASE1_EPOCHS = 5    # head only
PHASE2_EPOCHS = 15   # full fine-tune
PHASE1_LR     = 1e-3
PHASE2_LR     = 1e-3
VAL_SPLIT     = 0.2
RANDOM_SEED   = 42
NUM_CLASSES   = 2

In [ ]:
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
def load_img_data(file_path):
    with np.load(file_path) as data:
        imgs = data['images']
        ids  = data['image_ids']
    print(f"Loaded {imgs.shape[0]} images — dtype: {imgs.dtype}, "
          f"range: [{imgs.min()}, {imgs.max()}]")
    return imgs, ids


imgs,      img_ids      = load_img_data(IMG_FILE)
imgs_test, img_ids_test = load_img_data(IMG_TEST_FILE)

y_dev = pd.read_csv(LABEL_CSV)
print(f"\ny_dev shape: {y_dev.shape}")
print(y_dev.head())

Loaded 1178 images — dtype: uint8, range: [0, 255]
Loaded 296 images — dtype: uint8, range: [0, 255]

y_dev shape: (1178, 4)
  patient_id       img_id  coarse_label fine_label
0    PAT_620  PAT_620_001             1        BCC
1    PAT_388  PAT_388_002             1        BCC
2    PAT_518  PAT_518_003             1        BCC
3    PAT_635  PAT_635_004             1        BCC
4    PAT_447  PAT_447_005             1        ACK


In [ ]:
# ─────────────────────────────────────────────
# 2. ALIGN IMAGES WITH LABELS
# ─────────────────────────────────────────────
id_to_idx     = {img_id: i for i, img_id in enumerate(img_ids)}

y_aligned     = y_dev[y_dev['img_id'].isin(id_to_idx)].copy()
y_aligned['numpy_idx'] = y_aligned['img_id'].map(id_to_idx)
y_aligned     = y_aligned.sort_values('numpy_idx').reset_index(drop=True)

imgs_aligned  = imgs[y_aligned['numpy_idx'].values]
labels        = y_aligned['coarse_label'].values

print(f"\nAligned {len(labels)} samples")
print(f"Class distribution — 0: {(labels==0).sum()}, 1: {(labels==1).sum()}")


Aligned 1178 samples
Class distribution — 0: 89, 1: 1089


In [ ]:
# ─────────────────────────────────────────────
# 3. TRAIN / VAL SPLIT
# ─────────────────────────────────────────────
imgs_train, imgs_val, labels_train, labels_val = train_test_split(
    imgs_aligned, labels,
    test_size=VAL_SPLIT,
    random_state=RANDOM_SEED,
    stratify=labels          # preserve class ratio in both splits
)
print(f"\nTrain: {len(labels_train)}  |  Val: {len(labels_val)}")



Train: 942  |  Val: 236


In [ ]:
# ─────────────────────────────────────────────
# 4. DATASET & TRANSFORMS
# ─────────────────────────────────────────────
class LesionDataset(Dataset):
    def __init__(self, images, labels, transform=None, apply_geometric_augmentations=False):
        self.images    = images
        self.labels    = labels
        self.transform = transform
        self.apply_geometric_augmentations = apply_geometric_augmentations

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_array = self.images[idx].astype(np.uint8)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        pil_img_original = Image.fromarray(img_array)
        images_to_process = [pil_img_original]

        if self.apply_geometric_augmentations:
            # Convert PIL to NumPy for OpenCV operations
            cv_img = np.array(pil_img_original)
            h, w = cv_img.shape[:2]

            # Rotations (90, 180, 270 degrees)
            center = (w // 2, h // 2)
            M90 = cv2.getRotationMatrix2D(center, 90, 1.0)
            M180 = cv2.getRotationMatrix2D(center, 180, 1.0)
            M270 = cv2.getRotationMatrix2D(center, 270, 1.0)

            rot90_img = cv2.warpAffine(cv_img, M90, (w, h))
            rot180_img = cv2.warpAffine(cv_img, M180, (w, h))
            rot270_img = cv2.warpAffine(cv_img, M270, (w, h))

            # Convert back to PIL and add to list
            images_to_process.extend([
                Image.fromarray(rot90_img),
                Image.fromarray(rot180_img),
                Image.fromarray(rot270_img),
            ])

        # Apply the standard transform to each image in the list
        if self.transform:
            transformed_images = [self.transform(img) for img in images_to_process]
        else:
            # Fallback if no transform is provided
            transformed_images = [transforms.ToTensor()(img) for img in images_to_process]

        if self.apply_geometric_augmentations:
            return transformed_images, label # returns a list of tensors and a single label
        else:
            return transformed_images[0], label # returns a single tensor and a single label


# Custom collate_fn for handling batches when geometric augmentations are applied
def custom_collate_fn(batch):
    # 'batch' is a list of (list_of_transformed_images, label)
    # where list_of_transformed_images contains tensors

    all_images_in_batch = []
    all_labels_in_batch = []

    for transformed_imgs_list, label in batch:
        all_images_in_batch.extend(transformed_imgs_list)
        all_labels_in_batch.extend([label] * len(transformed_imgs_list)) # Repeat label for each augmented view

    # Stack all images into a single tensor (effective_batch_size, C, H, W)
    final_imgs_batch = torch.stack(all_images_in_batch)
    # Stack all labels into a single tensor (effective_batch_size)
    final_labels_batch = torch.stack(all_labels_in_batch)

    return final_imgs_batch, final_labels_batch


# ImageNet normalization stats — required for pretrained EfficientNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# MODIFIED: Instantiate train_dataset with geometric augmentations and use custom collate_fn
train_dataset = LesionDataset(imgs_train, labels_train,
                              transform=train_transform,
                              apply_geometric_augmentations=True) # Enable geometric augmentations

val_dataset   = LesionDataset(imgs_val,   labels_val,
                              transform=val_transform,
                              apply_geometric_augmentations=False) # No geometric augmentations for validation

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=NUM_WORKERS,
                           collate_fn=custom_collate_fn) # Use custom collate_fn

val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=NUM_WORKERS) # No custom collate_fn needed, as it returns single images

In [ ]:
# ─────────────────────────────────────────────
# 5. MODEL
# ─────────────────────────────────────────────
def build_model(num_classes=2, freeze_backbone=True):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace classifier head
    in_features = model.classifier[1].in_features   # 1280
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

model = build_model(num_classes=NUM_CLASSES, freeze_backbone=True).to(device)
print(f"\nTrainable params: "
      f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 152MB/s]



Trainable params: 2,562


In [ ]:
# ─────────────────────────────────────────────
# 6. LOSS — weighted for class imbalance
# ─────────────────────────────────────────────
class_counts  = torch.tensor(
    [(labels_train == c).sum() for c in range(NUM_CLASSES)], dtype=torch.float32
)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
print(f"Class weights: {class_weights}")

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))


Class weights: tensor([0.9246, 0.0754])


In [ ]:
# ─────────────────────────────────────────────
# 7. EVALUATION HELPER
# ─────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for imgs_batch, lbls in loader:
            imgs_batch = imgs_batch.to(device)
            outputs    = model(imgs_batch)
            probs      = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            preds      = torch.argmax(outputs, dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(lbls.numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    auc = roc_auc_score(all_labels, all_probs)
    print(f"  AUC: {auc:.4f}")
    return auc


In [ ]:
# ─────────────────────────────────────────────
# 8. TRAINING LOOP
# ─────────────────────────────────────────────
def run_epoch(model, loader, optimizer, criterion, training=True):
    model.train() if training else model.eval()
    total_loss = 0

    with torch.set_grad_enabled(training):
        for imgs_batch, lbls in loader:
            imgs_batch, lbls = imgs_batch.to(device), lbls.to(device)
            outputs = model(imgs_batch)
            loss    = criterion(outputs, lbls)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
# ── Phase 1: Head only ──────────────────────
print("\n" + "="*50)
print("PHASE 1 — Training classifier head only")
print("="*50)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=PHASE1_LR
)

for epoch in range(PHASE1_EPOCHS):
    train_loss = run_epoch(model, train_loader, optimizer, criterion, training=True)
    val_loss   = run_epoch(model, val_loader,   optimizer, criterion, training=False)
    print(f"\nEpoch {epoch+1}/{PHASE1_EPOCHS} — "
          f"Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")
    evaluate(model, val_loader)




PHASE 1 — Training classifier head only

Epoch 1/5 — Train Loss: 0.6259  Val Loss: 0.5103
  AUC: 0.8733

Epoch 2/5 — Train Loss: 0.5026  Val Loss: 0.4251
  AUC: 0.8978

Epoch 3/5 — Train Loss: 0.4504  Val Loss: 0.3958
  AUC: 0.9077

Epoch 4/5 — Train Loss: 0.4353  Val Loss: 0.3894
  AUC: 0.9207

Epoch 5/5 — Train Loss: 0.3797  Val Loss: 0.3604
  AUC: 0.9141


In [ ]:
# ── Phase 2: Full fine-tune ─────────────────
print("\n" + "="*50)
print("PHASE 2 — Full fine-tuning")
print("="*50)

for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=PHASE2_LR)

best_auc = 0.0
for epoch in range(PHASE2_EPOCHS):
    train_loss = run_epoch(model, train_loader, optimizer, criterion, training=True)
    val_loss   = run_epoch(model, val_loader,   optimizer, criterion, training=False)
    print(f"\nEpoch {epoch+1}/{PHASE2_EPOCHS} — "
          f"Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")
    auc = evaluate(model, val_loader)

    # Save best model checkpoint
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ✓ New best AUC: {best_auc:.4f} — model saved")


PHASE 2 — Full fine-tuning

Epoch 1/15 — Train Loss: 0.7098  Val Loss: 1.1189
  AUC: 0.8287
  ✓ New best AUC: 0.8287 — model saved

Epoch 2/15 — Train Loss: 0.3455  Val Loss: 0.3672
  AUC: 0.9266
  ✓ New best AUC: 0.9266 — model saved

Epoch 3/15 — Train Loss: 0.1957  Val Loss: 0.5543
  AUC: 0.9156

Epoch 4/15 — Train Loss: 0.1466  Val Loss: 0.6153
  AUC: 0.8767

Epoch 5/15 — Train Loss: 0.1137  Val Loss: 0.6667
  AUC: 0.9062

Epoch 6/15 — Train Loss: 0.1075  Val Loss: 0.7983
  AUC: 0.8940

Epoch 7/15 — Train Loss: 0.1327  Val Loss: 0.4176
  AUC: 0.9371
  ✓ New best AUC: 0.9371 — model saved

Epoch 8/15 — Train Loss: 0.1462  Val Loss: 0.9090
  AUC: 0.8902

Epoch 9/15 — Train Loss: 0.0911  Val Loss: 0.8384
  AUC: 0.8649

Epoch 10/15 — Train Loss: 0.0697  Val Loss: 1.5240
  AUC: 0.8490

Epoch 11/15 — Train Loss: 0.0412  Val Loss: 1.2103
  AUC: 0.8626

Epoch 12/15 — Train Loss: 0.0936  Val Loss: 0.5264
  AUC: 0.9197

Epoch 13/15 — Train Loss: 0.0785  Val Loss: 0.8319
  AUC: 0.9139

Epoch

In [ ]:
# # ─────────────────────────────────────────────
# # 9. FINAL TEST SET PREDICTIONS
# # ─────────────────────────────────────────────
# model.load_state_dict(torch.load("best_model.pth", map_location=device))

# test_dataset = LesionDataset(imgs_test,
#                              np.zeros(len(imgs_test), dtype=int),
#                              transform=val_transform)
# test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
#                           shuffle=False, num_workers=NUM_WORKERS)

# model.eval()
# test_probs = []

# with torch.no_grad():
#     for imgs_batch, _ in test_loader:
#         imgs_batch = imgs_batch.to(device)
#         outputs    = model(imgs_batch)
#         probs      = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
#         test_probs.extend(probs)

# # Write probabilities to file, one per line
# with open("yproba1_test_efficient_net.txt", "w") as f:
#     for pred in test_probs:
#         f.write(f"{pred}\n")

# print(f"Saved {len(test_probs)} predictions to yproba1_test.txt")

Saved 296 predictions to yproba1_test.txt


### Get embedding outputs from the neural network
To get the embedding outputs, we will load the best trained model, remove the classification head, and then pass the images through the modified model. The output before the classification head will be the embeddings.

In [ ]:
# Load the best model
model = build_model(num_classes=NUM_CLASSES, freeze_backbone=False).to(device)
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [ ]:
# Remove the classification head to get embeddings
embeddings_model = torch.nn.Sequential(*(list(model.children())[:-1]))
embeddings_model = embeddings_model.to(device)
embeddings_model.eval()

Sequential(
  (0): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivation(
    

In [ ]:
# Create a DataLoader for all aligned images to get embeddings
full_dataset = LesionDataset(imgs_aligned, labels, transform=val_transform)
full_loader  = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# Get embeddings for all images
all_embeddings = []

with torch.no_grad():
    for imgs_batch, _ in full_loader:
        imgs_batch = imgs_batch.to(device)
        embeddings = embeddings_model(imgs_batch)
        all_embeddings.append(embeddings.cpu().numpy())

embeddings_output = np.concatenate(all_embeddings, axis=0)
print(f"Shape of extracted embeddings: {embeddings_output.shape}")

Shape of extracted embeddings: (1178, 1280, 1, 1)


In [ ]:
from sklearn.decomposition import PCA

# Flatten the embeddings from (N, 1280, 1, 1) to (N, 1280)
flattened_embeddings = embeddings_output.reshape(embeddings_output.shape[0], -1)

# Initialize PCA. You can specify n_components, e.g., n_components=64, or keep it for variance explanation.
pca = PCA(n_components=0.95) # Retain 95% of variance
reduced_embeddings = pca.fit_transform(flattened_embeddings)

print(f"Original embedding shape: {embeddings_output.shape}")
print(f"Flattened embedding shape: {flattened_embeddings.shape}")
print(f"Reduced embedding shape (retaining 95% variance): {reduced_embeddings.shape}")
print(f"Number of components selected by PCA: {pca.n_components_}")

# Save embeddings to a file
np.save("image_embeddings.npy", reduced_embeddings)
print("Embeddings saved to image_embeddings.npy")

Original embedding shape: (1178, 1280, 1, 1)
Flattened embedding shape: (1178, 1280)
Reduced embedding shape (retaining 95% variance): (1178, 293)
Number of components selected by PCA: 293
Embeddings saved to image_embeddings.npy


### Get embeddings for test images

In [ ]:
# Create a DataLoader for test images to get embeddings
test_embedding_dataset = LesionDataset(imgs_test,
                                     np.zeros(len(imgs_test), dtype=int),
                                     transform=val_transform)
test_embedding_loader  = DataLoader(test_embedding_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
# Get embeddings for all test images
all_test_embeddings = []

with torch.no_grad():
    for imgs_batch, _ in test_embedding_loader:
        imgs_batch = imgs_batch.to(device)
        embeddings = embeddings_model(imgs_batch)
        all_test_embeddings.append(embeddings.cpu().numpy())

test_embeddings_output = np.concatenate(all_test_embeddings, axis=0)
print(f"Shape of extracted test embeddings: {test_embeddings_output.shape}")

Shape of extracted test embeddings: (296, 1280, 1, 1)


In [ ]:
# Flatten the embeddings from (N, 1280, 1, 1) to (N, 1280)
test_flattened_embeddings = test_embeddings_output.reshape(test_embeddings_output.shape[0], -1)

test_reduced_embeddings = pca.fit_transform(test_flattened_embeddings)

print(f"Original embedding shape: {test_embeddings_output.shape}")
print(f"Flattened embedding shape: {test_flattened_embeddings.shape}")
print(f"Reduced embedding shape (retaining 95% variance): {test_reduced_embeddings.shape}")
print(f"Number of components selected by PCA: {pca.n_components_}")

# Save test embeddings to a file
np.save("test_image_embeddings.npy", test_reduced_embeddings)
print("Test embeddings saved to test_image_embeddings.npy")

Original embedding shape: (296, 1280, 1, 1)
Flattened embedding shape: (296, 1280)
Reduced embedding shape (retaining 95% variance): (296, 141)
Number of components selected by PCA: 141
Test embeddings saved to test_image_embeddings.npy
